In [1]:
#%pip install spotipy
import spotipy 

from spotipy.oauth2 import SpotifyClientCredentials
import json 
from bs4 import BeautifulSoup #pip install beautifulsoup4
import requests

# Web Scrapping

Hoje quero mostrar pra vocês algumas possibilidades bem básicas para __extrair__ dados de sites e serviços na Web. Estes dados poderão ser usados para fazer análises, processamento, gerar novos serviços e etc. O objetivo é abrir os horizontes.

Vamos ver 3 tipos de Scrapping:
 - Scrapping via API
 - Scrapping via lib (sem API diretamente)
 - Scrapping via coletores Web

Nos próximos momentos do projeto vou mostrar coisas legais que dá para fazer com esses dados.


## Scrapping via API

Hoje em dia a vida do cientista de dados é muito mais fácil que antigamente, quando tudo era mato. Hoje, muitos serviços disponibilizam uma API de acesso para se fazer consultas, extrair dados e mmto mais. No ano anterior, os bolsistas do SUPER tiveram a chance de usar a API do Twitter. Hoje, mostrarei como exemplo o acesso a outra API, a do _Spotify_. Faremos acesso usando uma biblioteca wrapper, a Spotipy.

### Spotify

Antes de mais nada, acessem https://developer.spotify.com/dashboard/ e criem/vinculem a conta de vocês para desenvolvedores.

Depois de pronto, ao chegar no Dashboard, vamos criar um novo App. Como exemplo aqui, vamos fazer um aplicativo que colete informações sobre podcasts.

Com o App pronto, o que é importante é a chave (client ID) esse valor Hexadecimal de 32 caracteres é o que identifica o seu APP no Spotify e o segredo, que é uma senha relacionada a este ID, então mantenha ele em segredo (nada de deixar no código e subir no github).

Legal, com tudo pronto, agora baixem a biblioteca spotipy e rodem a primeira célula desse notebook. A gente vai salvar a chave em um arquivo json em separado, para não ter nossa chave em texto puro no meio do código. Criem um arquivo json na mesma pasta chamado auth.json com um conteúdo tipo assim:

`{"client_id": "5e9a80618b284145b54bb1f7df94bb6c", 
"client_secret": "0cdef7160e4143118e48abdd939668e8"}`

Vamos fazer um teste rápido para ver se está funcionando:

In [2]:
cred = json.load(open('auth.json'))
client_id = cred['client_id']
client_secret = cred['client_secret']

In [13]:
ccm = SpotifyClientCredentials(client_id=client_id,client_secret=client_secret)

sp = spotipy.Spotify(client_credentials_manager=ccm)

### Exemplo 1 - Pegando albuns de um dado artista:

In [4]:
uri = 'spotify:artist:1b8kpp4DUwt1hWaxTiWQhD'

results = sp.artist_albums(uri, album_type='album')
albums = results['items']
while results['next']:
    results = sp.next(results)
    albums.extend(results['items'])

for album in albums:
    print(album['name'])
    


Nenhuma Dor
A Pele do Futuro Ao Vivo
A Pele do Futuro
Trinca de Ases (Ao Vivo)
Estratosférica Ao Vivo
Gal Estratosférica
A Arte De Gal Costa
Recanto Ao Vivo
Recanto
Divina, Maravilhosa
Live at the Blue Note
Gal Costa (Ao Vivo)
Hoje
Divino Maravilhoso - Gal Costa Interpreta Caetano Veloso (CD 2)
Divino Maravilhoso - Gal Costa Interpreta Caetano Veloso (CD 1)
Gal Canta Caetano
O Amor
Todas as coisas e eu
Gal Bossa Tropical
Duetos
Gal De Tantos Amores
Gal Costa Sem Limite
Minha Voz, Minha Vida
T.S.O. Do Filme Gabriela
Gal Costa Canta Tom Jobim
Aquele Frevo Axé
Obras-Primas
Mina D'Água do Meu Canto
O Sorriso Do Gato De Alice
Plural
Rio Revisited
Profana
Baby Gal
Minha Voz
Fantasia
Aquarela Do Brasil
Gal Tropical
Agua Viva
Caras E Bocas
Gal Canta Caymmi
Temporada De Verao
Cantar
India
Live In London '71, Vol. 2 (Ao Vivo)
Live In London '71, Vol. 1 (Ao Vivo)
Gal A Todo Vapor (Live)
Legal
Gal Costa
Gal Costa
Domingo


In [10]:
results['artists']

{'href': 'https://api.spotify.com/v1/search?offset=0&limit=10&query=artist%3Araul%20seixas&type=artist',
 'limit': 10,
 'next': None,
 'offset': 0,
 'previous': None,
 'total': 2,
 'items': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/7jrRQZg4FZq6dwpi3baKcu'},
   'followers': {'href': None, 'total': 2382494},
   'genres': ['mpb',
    'psicodelia brasileira',
    'rock baiano',
    'rock nacional brasileiro'],
   'href': 'https://api.spotify.com/v1/artists/7jrRQZg4FZq6dwpi3baKcu',
   'id': '7jrRQZg4FZq6dwpi3baKcu',
   'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5eb9f0c70d55c26e9773d00b7b1',
     'height': 640,
     'width': 640},
    {'url': 'https://i.scdn.co/image/ab676161000051749f0c70d55c26e9773d00b7b1',
     'height': 320,
     'width': 320},
    {'url': 'https://i.scdn.co/image/ab6761610000f1789f0c70d55c26e9773d00b7b1',
     'height': 160,
     'width': 160}],
   'name': 'Raul Seixas',
   'popularity': 65,
   'type': 'artist',
   'uri': 'spotify:art

In [8]:
nome = "raul seixas"

results = sp.search(q='artist:' + nome, type='artist')
items = results['artists']['items']
if len(items) > 0:
    artist = items[0]
    print(artist['name'], artist['images'][0]['url'])

Raul Seixas https://i.scdn.co/image/ab6761610000e5eb9f0c70d55c26e9773d00b7b1


In [ ]:
playlists = sp.user_playlists('spotify')
while playlists:
    for i, playlist in enumerate(playlists['items']):
        print("%4d %s %s" % (i + 1 + playlists['offset'], playlist['url'],  playlist['name']))
    if playlists['next']:
        playlists = sp.next(playlists)
    else:
        playlists = None

TypeError: 'NoneType' object is not subscriptable

In [17]:
playlists = sp.user_playlists('spotify')
playlists

{'href': 'https://api.spotify.com/v1/users/spotify/playlists?offset=0&limit=50',
 'limit': 50,
 'next': 'https://api.spotify.com/v1/users/spotify/playlists?offset=50&limit=50',
 'offset': 0,
 'previous': None,
 'total': 1519,
 'items': [None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None]}

### Acessando diretamente pelo endpoint via GET

O spotipy é apenas uma biblioteca wrapper para facilitar a vida. Na prática, a mágica está no fato que o Spotify disponibiliza um endpoint REST que é acessável via HTTP, onde você manda uma requisição e recebe como resposta um arquivo JSON.

In [18]:
#Autorizacao

r = requests.post('https://accounts.spotify.com/api/token', {
    'grant_type':'client_credentials',
    'client_id' : client_id,
    'client_secret': client_secret
})

token = r.json()['access_token']

In [20]:
busca = 'videogames'
endpoint = 'http://api.spotify.com/v1/search?'
tipo = 'show'
market='BR'
lang='pt'
limite = '50'

cons = endpoint+'&q='+busca+'&type='+tipo+'&market='+market+'&limit='+limite+'&languages='+lang #+'&offset=50'
cons

'http://api.spotify.com/v1/search?&q=videogames&type=show&market=BR&limit=50&languages=pt'

In [21]:
r = requests.get(cons,headers={'Content-Type':'application/json',"Authorization":f"Bearer {token}"})
res = r.json()

In [22]:
res['shows']['items']

[{'copyrights': [],
  'description': 'Um podcast semanal sobre nostalgia e videogames que marcaram as nossas infâncias dos anos 80 e 90.',
  'html_description': 'Um podcast semanal sobre nostalgia e videogames que marcaram as nossas infâncias dos anos 80 e 90.',
  'explicit': False,
  'external_urls': {'spotify': 'https://open.spotify.com/show/4QhlBbsexWu8siiSJnv0iA'},
  'href': 'https://api.spotify.com/v1/shows/4QhlBbsexWu8siiSJnv0iA',
  'id': '4QhlBbsexWu8siiSJnv0iA',
  'images': [{'height': 640,
    'url': 'https://i.scdn.co/image/ab6765630000ba8add8d02e99e16e6c7037d2891',
    'width': 640},
   {'height': 300,
    'url': 'https://i.scdn.co/image/ab67656300005f1fdd8d02e99e16e6c7037d2891',
    'width': 300},
   {'height': 64,
    'url': 'https://i.scdn.co/image/ab6765630000f68ddd8d02e99e16e6c7037d2891',
    'width': 64}],
  'is_externally_hosted': False,
  'languages': ['pt-BR'],
  'media_type': 'audio',
  'name': '99Vidas - Nostalgia e Videogames',
  'publisher': '99Vidas',
  'type':

In [23]:
for i in res['shows']['items']:
    print(i['name'])

99Vidas - Nostalgia e Videogames
Galinha Viajante
Matando Robôs Gigantes
UP
PeeWeeCast
Fora do Controle
Obsessões Passageiras
Jogabilidade
Toca Do Dragão
Flow Games
Dinotronic
RapaduraCast - Podcast de Cinema e Streaming
Matei o Chefe
Frango Fino
The So Videogames Podcast
O X do Controle
Godmode Podcast
Jogo Véio Podcast
Sometimes Videogames
Cinemático
This Week In Videogames
Talking to Women about Videogames
Metal Gear Of Videogames Podcast
Debug Mode
The Jeremy VideoGames Video Game Hour
RELOADING - Atualize-se, gamer!
Vai Logar Hoje?
Controles Voadores
MeuPlayStation
Regras do Jogo - Holodeck
Pouco Pixel
Buraco Atrás do Pôster
mídias sonoras que eu preciso guardar lol
Velho Gamer - Informação e Nostalgia em Videogames
MotherChip - Overloadr
Insert Coin. Il grande gioco dei videogames
Séries Maniacos
The Radish & Bunday Retro Videogames Podcast
Player 1 
Código do Caos
Fabuloso Podcast
Sugarpulp Podcast: raccontiamo un mondo fatto di libri, fumetti, eventi, serie tv, film e videogame

### Sem API, sem biblioteca, sem nada

As vezes, o site simplesmente não quer liberar suas informações facilmente assim para nós, pobres desenvolvedores. Pra resolver isso, vamos usar algumas ferramentas para capturar NA MARRA informações disponíveis na Web.

Os passos que iremos seguir:
1. Encontrar a URL que vamos pegar
2. Inspecionar o HTML da página
3. Achar o dado que queremos
4. Programar :-)


Usaremos a biblioteca BeatifulSoup para ler o HTML e extrair os dados.
 
Como exemplo, vamos fazer um scrapper para descobrir qual o preço médio do playstation 5 no mercado livre.

Depois de achar, vamos inspecionar o preço para acharmos a tag HTML onde ele está.

De posse disso, vamos tentar extrair o nome do anúncio, preço e se tem frete grátis.

In [24]:

nomes = []
precos = []
frete = []
url = 'https://games.mercadolivre.com.br/consoles/playstation-5/ps5_NoIndex_True'
url2 = 'https://games.mercadolivre.com.br/consoles/playstation-5/ps5_Desde_49_NoIndex_True'
r= requests.get(url)
r2= requests.get(url2)



In [25]:
soup = BeautifulSoup(r.content)

for i in soup.findAll('div',attrs={'class':'ui-search-result__content-wrapper'}):
    print(i.find('span',attrs={'class':'price-tag-fraction'}).text)

In [26]:
soup2 = BeautifulSoup(r2.content)
for i in soup2.findAll('div',attrs={'class':'ui-search-result__content-wrapper'}):
    if(i.find('p',attrs={'ui-search-item__shipping ui-search-item__shipping--free'})):
        print(i.find('h2',attrs={'class':'ui-search-item__title ui-search-item__group__element'}).text)

Façam da descrição e do Frete Gratis (True/False).

Show de bola né? Vamos fazer uma leve automação? Vamos ver se conseguimos identificar o botão "seguinte" no HTML e fazer uma requisição para ele:

In [27]:
soup.find('a',href=True,attrs={'title':'Seguinte'})['href']

'https://lista.mercadolivre.com.br/games/consoles/playstation-5/ps5_Desde_49_NoIndex_True'

In [30]:

url = 'https://games.mercadolivre.com.br/consoles/playstation-5/ps5_NoIndex_True'
while url:
    r = requests.get(url)
    soup = BeautifulSoup(r.content)
    for i in soup.findAll('div',attrs={'class':'poly-price__current'}):
        precos.append(float(i.find('span',attrs={'class':'andes-money-amount'}).text.replace('.','').replace("R$","")))
    
    url = soup.find('a',href=True,attrs={'title':'Seguinte'})
    if url:
        url = url['href']
        
print(precos)

[3700.0, 3999.0, 6550.0, 6999.0, 4399.0, 3799.0, 4911.0, 4052.0, 3782.0, 3699.0, 3999.0, 4596.0, 4455.0, 3685.0, 4500.0, 4324.0, 9999.0, 10900.0, 5679.0, 4289.0, 4099.0, 4099.0, 3499.0, 10599.0, 4630.0, 4999.0, 3948.0, 9999.0, 4499.0, 4002.0, 4558.0, 7499.0, 3899.0, 4400.0, 4799.0, 4238.0, 4199.0, 4498.0, 4175.0, 4499.0, 5997.0, 4299.0, 4239.0, 5465.0, 6190.0, 4221.0, 3899.0, 4099.0, 4949.0, 3999.0, 3799.0, 4558.0, 3200.0, 3200.0, 4156.0, 4149.0, 7499.0, 8299.0, 7839.0, 4484.0, 4205.0, 5937.0, 4679.0, 12500.0, 4499.0, 4499.0, 7899.0, 4799.0, 4248.0, 4099.0, 4264.0, 3611.0, 4539.0, 4190.0, 4199.0, 5279.0, 7899.0, 3733.0, 6299.0, 7699.0, 4099.0, 8999.0, 5081.0, 4900.0, 4290.0, 4448.0, 5183.0, 3999.0, 4299.0, 3949.0, 4249.0, 4269.0, 4680.0, 5181.0, 3599.0, 4800.0, 8199.0, 4300.0, 4298.0, 4699.0, 4699.0, 10599.0, 4042.0, 10000.0, 3450.0, 3690.0, 3799.0, 220.0, 3869.0, 2800.0, 2385.0, 3099.0, 125.0, 5750.0, 89.0, 359.0, 2000.0, 3250.0, 3600.0, 3649.0, 3099.0, 5422.0, 335.0, 4399.0, 4679.0, 

Uma coisa que vocês podem ter percebido é que tem coisas aí no meio que claramente não são PS5. E agora, como retirar? E como fazer para calcular a média dos preços?

In [31]:
l = [i for i in precos if i>2000]
sum(l)/len(l)

4990.130081300813

In [32]:
min(l)

2385.0

In [ ]:
https://github.com/public-apis/public-apis